# Multi-controlled U gate (mini-project 3)

$\textbf{Question:}$ Write a Qiskit function that takes two inputs: a positive integer $n$ and a matrix $U \in \text{U}(2)$ and outputs a quantum circuit on $n + 1$ qubits, possibly with further ancillas, that implements a multi-controlled $U$ gate, $C^nU$, that is

$$C^nU|x\rangle_n|y\rangle_1 = \begin{cases} 
|x\rangle_n U|y\rangle_1, & \text{if } x = (1,1,\dots,1), \\ 
|x\rangle_n|y\rangle_1, & \text{otherwise.} 
\end{cases}$$

The construction may use any number of ancillas, arbitrary 1-qubit gates, $CX$, and Toffoli gates. No classical bit and measurements allowed.

In [22]:
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, AncillaRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.circuit.library import HGate, RZGate
from qiskit.circuit.library import XGate
from qiskit.circuit.library import RXGate
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector


We first want to define a function that takes an integer $n$ for the number of control qubits and a $2\times 2$ unitary matrix $U$ representing the single qubit gate to apply. The function outputs a QuantumCircuit, using 

* Control register c of size n
* Target register t of size 1
* Ancilla register a of size n-1 (when n>= 2)

For $n = 1$ it returns the usual controlled $U$. For $n >= 2$, it first computes a flag into the last ancilla using a short Toffoli ladder, applies $U$ controlled on that flag, and then uncomputes the ancillas back to $|0\cdots0\rangle$. The result then becomes $C^n(U)$ with clean ancillas.

In [5]:
def multi_controlled_U(n: int, U) -> QuantumCircuit:
    if not isinstance(n, int) or n < 1:
        raise ValueError("n must be a positive integer greater than or equal to 1.")

    U = np.asarray(U, dtype=complex)
    if U.shape != (2, 2):
        raise ValueError("U must be a 2x2 matrix.")
    if not np.allclose(U.conj().T @ U, np.eye(2), atol=1e-6):
        raise ValueError("U must be unitary.")

    c = QuantumRegister(n, "c")
    t = QuantumRegister(1, "t")
    a = AncillaRegister(max(0, n - 1), "a") 

    qc = QuantumCircuit(c, t, a, name=f"C^{n}U")
    U1 = UnitaryGate(U, label="U")

    if n == 1:
        qc.append(U1.control(1), [c[0], t[0]])
        return qc

    # a[0] = c0 & c1
    qc.ccx(c[0], c[1], a[0])
    # a[k-1] = c[k+1] & a[k-2]  for k=1..n-2
    for k in range(2, n):
        qc.ccx(c[k], a[k - 2], a[k - 1])

    flag = a[n - 2] 

    # Apply U controlled on the flag 
    qc.append(U1.control(1), [flag, t[0]])

    # Uncompute ancillas to reverse the ladder
    for k in range(n - 1, 1, -1):
        qc.ccx(c[k], a[k - 2], a[k - 1])
    qc.ccx(c[0], c[1], a[0])

    return qc


As a simple test case, we choose the Hadamard gate $H$ for the single-qubit unitary $U$. We then build the multi-controlled gate $C^7(H)$ in the following

In [9]:
U = HGate().to_matrix()
qc = multi_controlled_U(7, U)

print(qc.draw("text"))
print("total number of qubits =", qc.num_qubits)

                                                                      
c_0: ──■───────────────────────────────────────────────────────────■──
       │                                                           │  
c_1: ──■───────────────────────────────────────────────────────────■──
       │                                                           │  
c_2: ──┼────■─────────────────────────────────────────────────■────┼──
       │    │                                                 │    │  
c_3: ──┼────┼────■───────────────────────────────────────■────┼────┼──
       │    │    │                                       │    │    │  
c_4: ──┼────┼────┼────■─────────────────────────────■────┼────┼────┼──
       │    │    │    │                             │    │    │    │  
c_5: ──┼────┼────┼────┼────■───────────────────■────┼────┼────┼────┼──
       │    │    │    │    │                   │    │    │    │    │  
c_6: ──┼────┼────┼────┼────┼────■─────────■────┼────┼────┼────┼────┼──
      

The following function prepares an input state $|x\rangle|y\rangle$, simulates multi_controlled_U(n, U) with Statevector, and prints the nonzero output amplitudes.


In [15]:
def test_function(n, U, x_bits, y_bit):
    qc = multi_controlled_U(n, U)

    prep = QuantumCircuit(qc.num_qubits)
    for i, b in enumerate(x_bits):
        if b == "1":
            prep.x(i)
    if y_bit == 1:
        prep.x(n)

    out = Statevector.from_label("0"*qc.num_qubits).evolve(prep.compose(qc))

    # Print nonzero amplitudes
    print(f"Input controls={x_bits}, target={y_bit}")
    for idx, amp in enumerate(out.data):
        if abs(amp) > 1e-6:
            bits = format(idx, f"0{out.num_qubits}b")[::-1]
            print(f"amplitude={amp:.3f} on |{bits}>")
    print()


In the following we test a few basis inputs for $n=1,2,3$. When $x=11\cdots1$, the Hadamard operator $H$ is applied to the target; otherwise the state should remain unchanged. Below, we observe all ancillas cleared back to zero. 

In [17]:
U = HGate().to_matrix()

test_function(1, U, "1", 0) 
test_function(2, U, "11", 0)
test_function(2, U, "10", 0)
test_function(3, U, "111", 0)
test_function(3, U, "110", 0)
test_function(4, U, "1111", 0)
test_function(4, U, "1100", 0)


Input controls=1, target=0
amplitude=0.707-0.000j on |10>
amplitude=0.707-0.000j on |11>

Input controls=11, target=0
amplitude=0.707-0.000j on |1100>
amplitude=0.707-0.000j on |1110>

Input controls=10, target=0
amplitude=1.000+0.000j on |1000>

Input controls=111, target=0
amplitude=0.707-0.000j on |111000>
amplitude=0.707-0.000j on |111100>

Input controls=110, target=0
amplitude=1.000+0.000j on |110000>

Input controls=1111, target=0
amplitude=0.707-0.000j on |11110000>
amplitude=0.707-0.000j on |11111000>

Input controls=1100, target=0
amplitude=1.000+0.000j on |11000000>



Now let's examine the $R_x$-gate

In [23]:
U = RXGate(np.pi/2).to_matrix()

test_function(1, U, "1", 0)
test_function(2, U, "11", 0)
test_function(2, U, "10", 0)
test_function(3, U, "111", 0)
test_function(3, U, "110", 0)
test_function(4, U, "1111", 0)
test_function(4, U, "1100", 0)

Input controls=1, target=0
amplitude=0.707-0.000j on |10>
amplitude=-0.000-0.707j on |11>

Input controls=11, target=0
amplitude=0.707-0.000j on |1100>
amplitude=-0.000-0.707j on |1110>

Input controls=10, target=0
amplitude=1.000-0.000j on |1000>

Input controls=111, target=0
amplitude=0.707-0.000j on |111000>
amplitude=-0.000-0.707j on |111100>

Input controls=110, target=0
amplitude=1.000-0.000j on |110000>

Input controls=1111, target=0
amplitude=0.707-0.000j on |11110000>
amplitude=-0.000-0.707j on |11111000>

Input controls=1100, target=0
amplitude=1.000-0.000j on |11000000>

